# GSSS_010 - One Agent vs a Team of Agents

We give **one** agent a big toolbox and a job with **7 parts**. It fumbles - picks the wrong
tool, skips parts, stops early. Then we split the same job across a **supervisor + 4
specialists**, each holding only 2-4 tools, and it comes back complete.

**The job ("Launch Kit"):** from a one-line product idea, produce
1. a tagline + 3 benefit bullets
2. a product **image**
3. a **comparison table** vs a named competitor (needs a web lookup)
4. a 5-post launch thread
5. 8 hashtags
6. a tone check ("professional, not hypey?")
7. an "hours until launch" countdown

Same brief, same tools, same models - only the **architecture** changes.


## 0  Setup

In [ ]:
%pip install -q langchain langchain-groq langchain-openai langchain-community \
    groq ddgs requests pillow

In [ ]:
import os, re, json, time
from getpass import getpass

def ask(label, required=True):
    if os.getenv(label):
        return os.environ[label]
    return getpass(f"{label}{'' if required else '  (optional, Enter to skip)'}: ").strip()

GROQ_API_KEY       = ask("GROQ_API_KEY")
OPENROUTER_API_KEY = ask("OPENROUTER_API_KEY", required=False)

from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

CHAT_MODELS = [
    ChatGroq(model="qwen/qwen3.8-27b", api_key=GROQ_API_KEY, temperature=0, max_retries=3, max_tokens=3000),
    ChatGroq(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY, temperature=0, max_retries=2, max_tokens=3000),
]
if OPENROUTER_API_KEY:
    CHAT_MODELS.append(ChatOpenAI(model="minimax/minimax-m2.7:free",
        base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY, temperature=0, max_retries=2))

def plain_llm(prompt: str) -> str:
    """One text completion. Tries each model; if all are rate-limited, waits and retries."""
    err = None
    for wait in (0, 15, 30):
        if wait:
            time.sleep(wait)
        for m in CHAT_MODELS:
            try:
                return m.invoke(prompt).content
            except Exception as e:
                err = e
    raise err

print("models:", [getattr(m, "model_name", getattr(m, "model", "?")) for m in CHAT_MODELS])
print(plain_llm("Reply with one word: ready"))

## 1  The toolbox (14 tools)

Many of these are **deliberately confusable** - six different "write_*" tools, three
"search/lookup", etc. Only some are needed for the Launch Kit; a few (`write_blog_post`,
`write_email`, `summarize_text`, `translate_text`) are decoys for this job.

In [ ]:
from langchain_core.tools import tool
import requests

IMAGES = []   # image files produced during the current run (cleared before each agent)

def _mini(prompt, max_tokens=500):
    return plain_llm(prompt)

@tool
def web_search(query: str) -> str:
    """Search the web for current facts, companies, products, prices. Returns snippets + URLs."""
    try:
        from ddgs import DDGS
        hits = list(DDGS().text(query, max_results=4))
        return "\n".join(f"- {h['title']}: {h['body'][:160]}" for h in hits) or "no results"
    except Exception as e:
        return f"search error: {e}"

@tool
def wikipedia(topic: str) -> str:
    """Concise encyclopedic summary of a topic (person, place, concept, company)."""
    ua = {"User-Agent": "GSSS-Bot/1.0"}
    try:
        s = requests.get("https://en.wikipedia.org/w/api.php", headers=ua, timeout=15, params={
            "action": "query", "list": "search", "srsearch": topic, "format": "json", "srlimit": 1}).json()
        t = s["query"]["search"][0]["title"]
        r = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/" + t.replace(" ", "_"), headers=ua, timeout=15)
        return f"{t}: {r.json().get('extract', '')[:400]}"
    except Exception as e:
        return f"wikipedia error: {e}"

@tool
def write_tagline_and_benefits(product_description: str) -> str:
    """Write a punchy tagline plus exactly 3 benefit bullet points for a product."""
    return _mini(f"Write ONE short tagline and exactly 3 benefit bullets for this product. "
                 f"No preamble.\n\nPRODUCT: {product_description}")

@tool
def write_launch_thread(product_description: str, posts: int = 5) -> str:
    """Write a numbered social-media launch thread of N short posts for a product."""
    return _mini(f"Write a {posts}-post launch thread (numbered 1..{posts}, each 1-2 sentences) "
                 f"for this product. No preamble.\n\nPRODUCT: {product_description}")

@tool
def write_blog_post(topic: str) -> str:
    """Write a long-form blog article on a topic (400+ words)."""
    return _mini(f"Write a 400-word blog post about: {topic}")

@tool
def write_email(recipient: str, about: str) -> str:
    """Draft a formal email to someone about a subject."""
    return _mini(f"Write a short formal email to {recipient} about {about}.")

@tool
def summarize_text(text: str) -> str:
    """Summarise a passage of text into 2-3 sentences."""
    return _mini(f"Summarise in 2-3 sentences:\n{text}")

@tool
def translate_text(text: str, language: str) -> str:
    """Translate text into another language."""
    return _mini(f"Translate into {language}, output only the translation:\n{text}")

@tool
def check_tone(text: str) -> str:
    """Judge whether marketing text sounds professional vs hypey/salesy, and say why."""
    return _mini(f"Is this marketing copy professional or too hypey/salesy? One line verdict "
                 f"+ one line why.\n\n{text}")

@tool
def suggest_hashtags(text: str, n: int = 8) -> str:
    """Suggest N relevant, non-spammy hashtags for a piece of marketing text."""
    return _mini(f"Give exactly {n} relevant hashtags (space-separated, no explanation) for:\n{text}")

@tool
def comparison_table(our_product: str, competitor: str) -> str:
    """Build a short markdown comparison table between our product and a named competitor."""
    facts = web_search.invoke({"query": f"{competitor} features pricing"})
    return _mini(f"Build a markdown table comparing '{our_product}' vs '{competitor}' on EXACTLY "
                 f"4 rows: price, key feature, best for, weakness. Keep each cell under 15 words. "
                 f"Competitor notes:\n{facts}\n\nOutput ONLY the 4-row table.")

@tool
def generate_image(prompt: str) -> str:
    """Generate a product image from a text prompt. The image is saved and shown automatically."""
    try:
        url = "https://image.pollinations.ai/prompt/" + requests.utils.quote(prompt) + "?width=768&height=512&nologo=true"
        data = requests.get(url, timeout=90).content
        p = f"/tmp/img_{len(IMAGES)}_{int(time.time())}.png"
        open(p, "wb").write(data); IMAGES.append(p)
        return f"Image generated: {prompt}"
    except Exception as e:
        return f"image error: {e}"

@tool
def current_datetime() -> str:
    """The current date and time (UTC)."""
    import datetime
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

@tool
def hours_until(iso_datetime: str) -> str:
    """Hours from now until a given ISO datetime like '2026-09-05 09:00'."""
    import datetime
    try:
        target = datetime.datetime.fromisoformat(iso_datetime)
        now = datetime.datetime.now()
        return f"{(target - now).total_seconds() / 3600:.1f} hours until {iso_datetime}"
    except Exception as e:
        return f"date error: {e}"

ALL_TOOLS = [web_search, wikipedia, write_tagline_and_benefits, write_launch_thread,
             write_blog_post, write_email, summarize_text, translate_text, check_tone,
             suggest_hashtags, comparison_table, generate_image, current_datetime, hours_until]
print(len(ALL_TOOLS), "tools:", [t.name for t in ALL_TOOLS])

## 2  Warm-up: the toolbox is crowded

Give ONE agent all 14 tools. On a *single* simple ask, a capable model usually still picks
right - the real damage shows up later, on the multi-part job, where it wastes steps on
decoy tools and loses track of what's left to do. This cell just gets you looking at the
tool trace.

In [ ]:
from langchain.agents import create_agent

def run_agent(agent, prompt, max_steps=8):
    """Invoke an agent; return (final_text, [tool names it called]). Retries on rate limits."""
    msgs = [{"role": "user", "content": prompt}] if isinstance(prompt, str) else prompt
    err = None
    for wait in (0, 15, 30):
        if wait:
            time.sleep(wait)
        for ag in agent:
            try:
                out = ag.invoke({"messages": msgs}, {"recursion_limit": max_steps * 2 + 3})
                trace = [tc["name"] for m in out["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
                return out["messages"][-1].content, trace
            except Exception as e:
                err = e
    raise err

def make_agent(tools, system=None):
    return [create_agent(m, tools, system_prompt=system) for m in CHAT_MODELS]

big_agent = make_agent(ALL_TOOLS)

probes = [
    ("Turn this into one punchy tagline: 'a wristband that senses stress and reminds you to breathe'.",
     "write_tagline_and_benefits"),
    ("Give me 8 hashtags for a post about a new AI note-taking app.", "suggest_hashtags"),
    ("Draw a picture of a red electric scooter.", "generate_image"),
]
for q, expected in probes:
    IMAGES.clear()
    _, trace = run_agent(big_agent, q, max_steps=4)
    print(f"asked -> should use `{expected}`  |  agent called: {trace}")
print("\nHold that thought - now give it the whole 7-part job.")

## 3  The single agent takes the whole job

In [ ]:
BRIEF = (
    "Product: 'PulseBand' - a wristband that tracks stress and nudges you to breathe. "
    "Launching 2026-09-06 09:00. Main competitor: Fitbit. Audience: busy professionals."
)

CHECKLIST = [
    "tagline_and_benefits", "product_image", "comparison_table_vs_competitor",
    "launch_thread_5_posts", "hashtags", "tone_check", "hours_until_launch",
]

SINGLE_PROMPT = (
    "You are a marketing assistant. Produce a complete 'Launch Kit' for the product in the "
    "brief. Use tools as needed. Deliver everything in one final message.\n\nBRIEF:\n" + BRIEF
)

IMAGES.clear()
t0 = time.time()
single_answer, single_trace = run_agent(big_agent, SINGLE_PROMPT, max_steps=16)
single_time = time.time() - t0
single_images = list(IMAGES)

print("TOOLS THE SINGLE AGENT CALLED:", single_trace)
print(f"(images produced: {len(single_images)}, wall time: {single_time:.0f}s)\n")
print(single_answer[:2500])

In [ ]:
from IPython.display import Image, display
for p in single_images:
    display(Image(p))

### Score it against the 7-item checklist

An LLM judge marks each item **present / partial / missing** from the deliverable.

In [ ]:
def judge(deliverable, n_images):
    raw = plain_llm(
        "Score this Launch Kit against the checklist. For EACH key say only "
        "'present', 'partial' or 'missing'.\n"
        f"CHECKLIST KEYS: {CHECKLIST}\n\nKIT:\n{deliverable[:4000]}\n\n"
        'Return ONLY JSON like {"tagline_and_benefits":"present", ...} with all 7 keys.')
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        score = json.loads(m.group(0))
    except Exception:
        score = {k: "?" for k in CHECKLIST}
    score["product_image"] = "present" if n_images else "missing"   # the file exists or it doesn't
    return score

def show_score(name, score):
    got = sum(1 for v in score.values() if v == "present")
    print(f"\n{name}: {got}/7 present")
    for k in CHECKLIST:
        v = score.get(k, "?")
        mark = {"present": "[x]", "partial": "[~]", "missing": "[ ]"}.get(v, "[?]")
        print(f"  {mark} {k:32s} {v}")
    return got

single_score = judge(single_answer, len(single_images))
single_got = show_score("SINGLE AGENT", single_score)

## 4  Same job, a team

**Editor-in-Chief (supervisor)** - no tools, just plans and assembles - delegates to four
specialists, each with a short, focused toolbox:

| Specialist | Tools | Job |
|---|---|---|
| `researcher` | web_search, wikipedia | competitor facts, market context |
| `copywriter` | write_tagline_and_benefits, write_launch_thread, suggest_hashtags, check_tone | all the words |
| `designer` | generate_image | the product image |
| `analyst` | comparison_table, current_datetime, hours_until | the table + the countdown |

Nobody sees the decoy tools. Nobody has to juggle all 7 parts at once.

In [ ]:
researcher = make_agent([web_search, wikipedia],
    "You are a researcher. Do AT MOST 2 searches, then answer with concise sourced facts. No fluff.")
copywriter = make_agent([write_tagline_and_benefits, write_launch_thread, suggest_hashtags, check_tone],
    "You are a copywriter. Do exactly what the task asks using your tools. Return the finished copy.")
designer = make_agent([generate_image],
    "You are a designer. Call generate_image once with a vivid, specific prompt, then confirm.")
analyst = make_agent([comparison_table, current_datetime, hours_until],
    "You are an analyst. Use your tools for the table and the countdown. ALWAYS end your reply "
    "with a line exactly like:  COUNTDOWN: <number> hours until launch")

TEAM = {"researcher": researcher, "copywriter": copywriter, "designer": designer, "analyst": analyst}

def multi_agent(brief, verbose=True):
    plan_raw = plain_llm(
        "You are Editor-in-Chief. Split this brief into 4-6 delegated tasks for your team.\n"
        "TEAM: researcher (facts/competitor), copywriter (tagline+benefits, 5-post thread, "
        "hashtags, tone check), designer (one product image), analyst (comparison table vs "
        "competitor, hours until launch).\n\n"
        f"BRIEF: {brief}\n\n"
        'Return ONLY a JSON list: [{"agent": "copywriter", "task": "..."}, ...]')
    m = re.search(r"\[.*\]", plan_raw, re.S)
    plan = json.loads(m.group(0)) if m else []

    results = {}
    for step in plan:
        ag = TEAM.get(step.get("agent"))
        if not ag:
            continue
        text, trace = run_agent(ag, f"{step['task']}\n\n(Overall brief for context: {brief})", max_steps=5)
        results.setdefault(step["agent"], []).append(text)
        if verbose:
            print(f"  {step['agent']:11s} | {step['task'][:70]}")
            print(f"              | tools={trace}  ->  {len(text)} chars")

    team_text = "\n\n".join(f"[{k}]\n" + "\n".join(v) for k, v in results.items())

    def assemble(extra=""):
        return plain_llm(
            "Assemble the final Launch Kit. Output these 7 sections IN THIS ORDER, each with a "
            "heading:\n"
            "1. Tagline & 3 benefits\n2. Hashtags (one line)\n3. Tone check verdict (one line)\n"
            "4. Hours until launch (one line - copy the COUNTDOWN line from the analyst)\n"
            "5. Product image note (one line)\n6. Comparison table vs the competitor (<=5 rows)\n"
            "7. The 5-post launch thread\n"
            "Copy real values from the team output VERBATIM. Never skip a section."
            + extra + f"\n\nTEAM OUTPUT:\n{team_text}\n\nBRIEF: {brief}")

    assembled = assemble()
    gaps = plain_llm(f"Checklist: {CHECKLIST}\nList only the items MISSING from this kit "
                     f"(comma-separated), or 'none'.\n\n{assembled[:4000]}")
    if "none" not in gaps.lower()[:15]:
        if verbose:
            print("  supervisor  | gap-check: re-assembling to add ->", gaps[:120])
        assembled = assemble(f"\n\nThe first draft was MISSING: {gaps}. Those sections MUST appear this time.")

    # deterministic safety net: the analyst already computed the countdown - if the
    # assembly LLM dropped it, splice the analyst's COUNTDOWN line back in verbatim.
    if "countdown" not in assembled.lower():
        cd = re.search(r"COUNTDOWN:.*", "\n".join(results.get("analyst", [])), re.I)
        if cd:
            if verbose:
                print("  supervisor  | splicing analyst countdown back in")
            assembled += f"\n\n## Hours until launch\n{cd.group(0).strip()}"

    return assembled

IMAGES.clear()
t0 = time.time()
multi_answer = multi_agent(BRIEF)
multi_time = time.time() - t0
multi_images = list(IMAGES)
print(f"\n(images: {len(multi_images)}, wall time: {multi_time:.0f}s)\n")
print(multi_answer[:2800])

In [ ]:
for p in multi_images:
    display(Image(p))
multi_score = judge(multi_answer, len(multi_images))
multi_got = show_score("MULTI-AGENT TEAM", multi_score)

## 5  Side by side

In [ ]:
print(f"{'deliverable':34s} {'single agent':>14s} {'multi-agent':>14s}")
print("-" * 64)
for k in CHECKLIST:
    print(f"{k:34s} {single_score.get(k,'?'):>14s} {multi_score.get(k,'?'):>14s}")
print("-" * 64)
print(f"{'SCORE':34s} {str(single_got)+'/7':>14s} {str(multi_got)+'/7':>14s}")
print(f"{'tool calls':34s} {len(single_trace):>14d} {'(spread across 4 agents)':>14s}")
print(f"{'wall time (s)':34s} {single_time:>14.0f} {multi_time:>14.0f}")

## Recap

- **Why the single agent slipped:** 14 look-alike tools -> wrong picks; one 7-part job in one
  trajectory -> parts forgotten; one generic prompt -> nothing is done really well; no step
  that checks the result against the ask.
- **What the team changed:** each specialist sees **2-4 tools** and **one job**, so tool
  choice is easy and quality is higher; the **supervisor** owns the checklist and runs a
  **gap-check** before finishing.
- **The trade-off:** the team makes more model calls and takes longer. Use multi-agent when
  the task has **many distinct parts**, needs **different kinds of expertise**, or the
  toolbox is **large**. For a small, single-domain task, one agent is cheaper and fine.

### Exercises
1. Cut `ALL_TOOLS` down to the 8 tools this job actually needs and re-run the single agent.
   How much of the gap was just tool overload?
2. Remove the supervisor's gap-check step. Which checklist item slips first?
3. Add a 5th specialist (a `reviewer` with `check_tone`) and have the supervisor always end
   with it.
4. Swap the single agent's model to `openai/gpt-oss-20b`. Does a bigger toolbox hurt a
   smaller model more?
